# Etape 3 — Generation de rapport KYC

Ce notebook genere un rapport legal de verification d'identite a partir des resultats des etapes precedentes.

**Inputs recus :**
- Etape 1 : type de document + score de confiance
- Etape 2.1 : photo detectee (oui/non) + date d'expiration
- Etape 2.2 : resultat face match + score de similarite

**Output :** rapport legal en Markdown genere via OpenRouter

---
## 1. Installation

In [ ]:
!pip install openai -q
print('Installation terminee.')

---
## 2. Imports

In [ ]:
import json
from openai import OpenAI
from google.colab import userdata
from IPython.display import Markdown, display

print('Imports OK.')

---
## 3. Configuration OpenRouter

**Avant de lancer cette cellule :**
1. Creer un compte sur openrouter.ai
2. Recuperer votre cle API
3. Dans Colab, cliquer sur l'icone cle (secrets) dans la barre de gauche
4. Ajouter `OPENROUTER_KEY` avec votre cle API

In [ ]:
# Recuperation de la cle API depuis les secrets Colab
api_key = userdata.get('OPENROUTER_KEY')

client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=api_key,
)

print('OpenRouter configure.')

---
## 4. Resultats des etapes precedentes

Renseigner ici les resultats recus des etapes 1, 2.1 et 2.2.
Dans le pipeline complet ces valeurs seront passees automatiquement.

In [ ]:
# --- Resultats Etape 1 : Classification du document ---
type_document  = 'Carte Nationale d Identite'  # ex: Passeport, Permis de conduire
confiance_doc  = 0.99                           # score entre 0 et 1

# --- Resultats Etape 2.1 : Detection des elements cles ---
photo_detectee  = True           # True si la photo a ete detectee sur le document
date_expiration = '2032-12-14'   # format YYYY-MM-DD, 'Non detectee' si absent

# --- Resultats Etape 2.2 : Face Match ---
est_match       = True    # True = meme personne, False = personnes differentes
score_similarite = 87.3   # score en pourcentage (0-100)
confiance_match  = 'HAUTE' # HAUTE, MOYENNE ou FAIBLE
distance         = 0.248   # distance cosine

print('Resultats charges.')
print(f'  Document    : {type_document} ({confiance_doc:.0%})')
print(f'  Photo       : {"Detectee" if photo_detectee else "Non detectee"}')
print(f'  Expiration  : {date_expiration}')
print(f'  Face Match  : {"MATCH" if est_match else "NO MATCH"} ({score_similarite}%)')

---
## 5. Generation du rapport KYC

Le LLM recoit les resultats des etapes precedentes et genere un rapport legal structure.

In [ ]:
def determiner_statut(est_match, photo_detectee, date_expiration, confiance_doc):
    """
    Determine le statut KYC en fonction des resultats des etapes precedentes.
    Retourne : APPROUVE, REJETE ou REVUE MANUELLE
    """
    if not est_match:
        return 'REJETE'
    if not photo_detectee:
        return 'REJETE'
    if date_expiration == 'Non detectee':
        return 'REVUE MANUELLE'
    if confiance_doc < 0.70:
        return 'REVUE MANUELLE'
    return 'APPROUVE'


def generer_rapport(type_document, confiance_doc, photo_detectee,
                    date_expiration, est_match, score_similarite,
                    confiance_match, distance):
    """
    Genere un rapport legal KYC via OpenRouter.
    Retourne le rapport au format Markdown.
    """

    statut = determiner_statut(est_match, photo_detectee, date_expiration, confiance_doc)

    SYSTEM_PROMPT = """
Tu es un expert en conformite KYC (Know Your Customer).
Tu rediges des rapports de verification d'identite formels, factuels et juridiquement precis.
Ton langage est professionnel, objectif et oriente vers la decision finale.
Tu utilises le format Markdown avec les sections suivantes uniquement :

# RAPPORT DE VERIFICATION KYC

## 1. Resume Executif
## 2. Analyse du Document d'Identite
## 3. Verification Biometrique
## 4. Conclusion Legale

Regles strictes :
- Factuel uniquement, ne pas inventer d'informations
- Convertir les scores decimaux en pourcentages
- Utiliser uniquement les emojis : OK, ATTENTION, ERREUR pour les statuts
- Langage juridique et professionnel
"""

    prompt = f"""
Voici les resultats de la verification KYC a rapporter :

ETAPE 1 - Classification du document :
- Type de document : {type_document}
- Confiance de classification : {confiance_doc:.0%}

ETAPE 2.1 - Detection des elements cles :
- Photo sur le document : {'Detectee' if photo_detectee else 'Non detectee'}
- Date d'expiration : {date_expiration}

ETAPE 2.2 - Verification biometrique (Face Match) :
- Resultat : {'MATCH - Meme personne' if est_match else 'NO MATCH - Personnes differentes'}
- Score de similarite : {score_similarite}%
- Niveau de confiance : {confiance_match}
- Distance cosine : {distance} (seuil KYC : 0.30)

STATUT GLOBAL DETERMINE : {statut}
"""

    completion = client.chat.completions.create(
        extra_headers={
            'HTTP-Referer': 'https://colab.research.google.com',
            'X-Title': 'KYC AI Solutions',
        },
        model='openrouter/owl-alpha',
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt}
        ]
    )

    return completion.choices[0].message.content


# Generation du rapport
print('Generation du rapport en cours...')

rapport = generer_rapport(
    type_document    = type_document,
    confiance_doc    = confiance_doc,
    photo_detectee   = photo_detectee,
    date_expiration  = date_expiration,
    est_match        = est_match,
    score_similarite = score_similarite,
    confiance_match  = confiance_match,
    distance         = distance
)

print('Rapport genere.')

---
## 6. Affichage du rapport

In [ ]:
# Affichage du rapport en Markdown dans Colab
display(Markdown(rapport))

---
## 7. Sauvegarde du rapport

Le rapport est sauvegarde en deux formats : Markdown et JSON (pour l'etape 4).

In [ ]:
# Sauvegarde en Markdown
with open('rapport_kyc.md', 'w', encoding='utf-8') as f:
    f.write(rapport)
print('rapport_kyc.md sauvegarde.')

# Sauvegarde en JSON pour l'etape 4 (interface)
statut_final = determiner_statut(est_match, photo_detectee, date_expiration, confiance_doc)

resultat_json = {
    'statut'           : statut_final,
    'type_document'    : type_document,
    'confiance_doc'    : confiance_doc,
    'photo_detectee'   : photo_detectee,
    'date_expiration'  : date_expiration,
    'face_match'       : est_match,
    'score_similarite' : score_similarite,
    'rapport_markdown' : rapport
}

with open('rapport_kyc.json', 'w', encoding='utf-8') as f:
    json.dump(resultat_json, f, ensure_ascii=False, indent=2)

print('rapport_kyc.json sauvegarde.')
print(f'Statut final : {statut_final}')